In [1]:
from langgraph.graph import StateGraph, START , END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict

In [2]:
load_dotenv()

True

In [4]:
subgraph_llm = ChatGroq(model='openai/gpt-oss-120b')
parent_llm = ChatGroq(model='openai/gpt-oss-120b')

In [5]:
class subgraph_state(TypedDict):
    input_text: str
    translated_text : str

In [ ]:
class parent_state(TypedDict):
    question : str
    answer_eng : str
    answer_urdu : str

In [14]:
def translate_response(state: subgraph_state):
    """You have given a text , you need to translate it into URDU."""
    message = state['input_text']
    response = subgraph_llm.invoke(f'Translate this text into URDU : {message}').content

    return {'translated_text': response}

In [15]:
subgraph = StateGraph(subgraph_state)

subgraph.add_node('translate_response', translate_response)

subgraph.add_edge(START, 'translate_response')
subgraph.add_edge('translate_response', END)

subgraph_workflow = subgraph.compile()

In [ ]:
def translate_answer(state: parent_state):

    response = subgraph_workflow.invoke({'input_text':state['answer_eng']})
    return {'answer_urdu': response['translated_text']}

In [ ]:
def generate_text(state: parent_state):
    question = state['question']
    response = parent_llm.invoke(f'simple give the answer of it : {question}').content

    return {'answer_eng': response} 

In [ ]:
parentgraph = StateGraph(parent_state)

parentgraph.add_node('Generate_text', generate_text)
parentgraph.add_node('translate', translate_answer)

parentgraph.add_edge(START, 'Generate_text')
parentgraph.add_edge('Generate_text', 'translate')
parentgraph.add_edge('translate' , END)

parentgraph_workflow = parentgraph.compile()

In [34]:
parentgraph_workflow.invoke({'question':'who is the current PM of pakistan'})

{'question': 'who is the current PM of pakistan',
 'input_text': 'I’m sorry, but I don’t have up‑to‑date information beyond mid‑2024. As of my last update, the caretaker Prime Minister was **Anwaar‑ul‑Haq\u202fKakar**, and a new government was expected to take office after the 2024 general election. I don’t have reliable data on who is serving as Prime Minister in August\u202f2026. For the most current answer, I recommend checking a recent news source or an official government website.',
 'translated_text': 'معذرت کے ساتھ، میرے پاس 2024 کے وسط کے بعد کی تازہ ترین معلومات نہیں ہیں۔ میری آخری اپ ڈیٹ کے مطابق، کیئر ٹیکر وزیرِ اعظم **انورالحق کاکڑ** تھے، اور 2024 کے عام انتخابات کے بعد نئی حکومت کے دفتر سنبھالنے کی توقع تھی۔ میرے پاس اگست\u202f2026 میں کون وزیرِ اعظم کے طور پر خدمات سر انجام دے رہا ہے، اس کے بارے میں قابلِ اعتماد ڈیٹا نہیں ہے۔ سب سے تازہ جواب کے لیے، کسی حالیہ خبری ذرائع یا سرکاری ویب سائٹ کو چیک کرنے کی سفارش کرتا ہوں۔'}